# Prueba de normalización con spaCy + POS (5 documentos)

Prueba rápida antes de la Fase 2: se comparan dos variantes sobre los **5 primeros abstracts** del corpus.

- **Sin quitar stopwords**: tokenización + lematización (solo se quita la puntuación).
- **Quitando stopwords por POS**: además se eliminan `DET, ADP, CCONJ, SCONJ, PRON`, como pide el plan.

In [1]:
from pathlib import Path

import pandas as pd
import spacy

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
nlp = spacy.load("en_core_web_sm")
print("spaCy", spacy.__version__, "| modelo", nlp.meta["name"], nlp.meta["version"])

docs = pd.read_csv(ROOT / "data" / "scientific_raw_corpus.tsv", sep="\t", dtype=str,
                   keep_default_na=False).head(5)
docs[["arxiv_id", "primary_category", "title"]]

spaCy 3.8.16 | modelo core_web_sm 3.8.0


,arxiv_id,primary_category,title
0,2610.00840,cs.CL,Contextual trajectory and incremental contextu...
1,2610.00833,cs.CL,VERITYGATE: A Four-Gate Schema-Level Faithfuln...
2,2610.00827,cs.CL,Verbalized and Internal Probabilities Are Coup...
3,2610.06903,cs.CL,Component and Dimension Sparsity in Transforme...
4,2610.00795,cs.CL,Can large language models unlock discrete data...


## 1. Etiquetas POS de una frase

Primera oración del primer abstract, token por token.

In [2]:
first_sentence = next(nlp(docs.loc[0, "abstract"]).sents)
pd.DataFrame([(t.text, t.pos_, spacy.explain(t.pos_), t.lemma_) for t in first_sentence],
             columns=["token", "POS", "significado", "lema"])

,token,POS,significado,lema
0,Transformer,NOUN,noun,transformer
1,-,PUNCT,punctuation,-
2,based,VERB,verb,base
3,large,ADJ,adjective,large
4,language,NOUN,noun,language
5,models,NOUN,noun,model
6,(,PUNCT,punctuation,(
7,LLMs,PROPN,proper noun,LLMs
8,),PUNCT,punctuation,)
9,such,ADJ,adjective,such


## 2. Las dos variantes de normalización

In [3]:
# Etiquetas POS (universales de spaCy) que se consideran stopwords y se eliminan.
# Son palabras gramaticales: sirven para armar la frase, aparecen en casi todos
# los textos y no indican el tema del documento.
STOP_POS = {
    "DET",    # determinantes: the, a, an, this, each
    "ADP",    # preposiciones: of, in, on, with, from
    "CCONJ",  # conjunciones coordinantes: and, or, but
    "SCONJ",  # conjunciones subordinantes: that, because, if, while
    "PRON",   # pronombres: we, it, they, which
}
# Se conservan NOUN, PROPN, VERB y ADJ, que son las que dicen de qué trata el texto.


def normalize(text: str, remove_stopwords: bool = True) -> str:
    doc = nlp(text)
    return " ".join(
        t.lemma_.lower() for t in doc
        if not (t.is_punct or t.is_space) and not (remove_stopwords and t.pos_ in STOP_POS)
    )


docs["sin_quitar"] = docs["abstract"].map(lambda s: normalize(s, remove_stopwords=False))
docs["quitando_stopwords"] = docs["abstract"].map(normalize)

pd.set_option("display.max_colwidth", 200)
docs[["arxiv_id", "sin_quitar", "quitando_stopwords"]]

,arxiv_id,sin_quitar,quitando_stopwords
0,2610.00840,transformer base large language model llms such as roberta represent text use contextual word embedding cwes which alter the embedding associate with each token base on surround context we constru...,transformer base large language model llms such roberta represent text use contextual word embedding cwes alter embedding associate token base surround context construct token wise incremental tra...
1,2610.00833,fluent llm explanation may not follow the evidence from a structured system we present veritygate a four gate checker for declare evidence id entity number and claim type it check a fix schema it ...,fluent llm explanation may not follow evidence structured system present veritygate four gate checker declare evidence id entity number claim type check fix schema do not verify fact prose r=0 r=1...
2,2610.00827,large language model carry an internal notion of uncertainty in their sampling distribution i.e. the probability they place on generate one answer rather than another they can also be ask to state...,large language model carry internal notion uncertainty sampling distribution i.e. probability place generate one answer rather can also be ask to state confidence word number verbalize uncertainty...
3,2610.06903,activation steering manipulate large language model behavior by intervene on internal activation but the mechanistic basis of these intervention remains poorly understand we decompose refusal stee...,activation steering manipulate large language model behavior intervene internal activation mechanistic basis intervention remains poorly understand decompose refusal steering component level inter...
4,2610.00795,objective to assess the accuracy and efficiency of a large language model llm use two prompt strategy to extract structured datum from ophthalmic diagnostic pdf report method twenty deidentifie re...,objective to assess accuracy efficiency large language model llm use two prompt strategy to extract structured datum ophthalmic diagnostic pdf report method twenty deidentifie report four type vis...


## 3. Comparación

Cuántos tokens quedan, cuánto se reduce el texto y qué palabras se eliminaron.

In [4]:
from collections import Counter

docs["tokens_original"] = docs["abstract"].map(lambda s: sum(1 for t in nlp(s) if not (t.is_punct or t.is_space)))
docs["tokens_sin_quitar"] = docs["sin_quitar"].str.split().str.len()
docs["tokens_quitando"] = docs["quitando_stopwords"].str.split().str.len()
docs["reduccion_%"] = (100 * (1 - docs["tokens_quitando"] / docs["tokens_sin_quitar"])).round(1)
display(docs[["arxiv_id", "tokens_sin_quitar", "tokens_quitando", "reduccion_%"]])

removed = Counter(t.lemma_.lower() for s in docs["abstract"] for t in nlp(s) if t.pos_ in STOP_POS)
print("Palabras eliminadas más frecuentes (por POS):")
pd.DataFrame(removed.most_common(15), columns=["lema", "veces"])

,arxiv_id,tokens_sin_quitar,tokens_quitando,reduccion_%
0,2610.00840,195,146,25.1
1,2610.00833,199,141,29.1
2,2610.00827,203,131,35.5
3,2610.06903,169,119,29.6
4,2610.00795,244,192,21.3


Palabras eliminadas más frecuentes (por POS):


,lema,veces
0,and,35
1,a,25
2,the,23
3,for,20
4,of,18
5,we,16
6,in,15
7,that,10
8,by,9
9,with,8


## 4. Efecto en bigramas

Bigramas del primer documento en cada variante (los que se forman con stopwords suelen no tener significado).

In [5]:
def bigrams(text: str) -> list[str]:
    w = text.split()
    return [f"{a} {b}" for a, b in zip(w, w[1:])]


b_sin, b_con = bigrams(docs.loc[0, "sin_quitar"]), bigrams(docs.loc[0, "quitando_stopwords"])
print(f"Bigramas distintos sin quitar: {len(set(b_sin))} | quitando stopwords: {len(set(b_con))}")
pd.DataFrame({"sin_quitar": b_sin[:12], "quitando_stopwords": b_con[:12]})

Bigramas distintos sin quitar: 184 | quitando stopwords: 137


,sin_quitar,quitando_stopwords
0,transformer base,transformer base
1,base large,base large
2,large language,large language
3,language model,language model
4,model llms,model llms
5,llms such,llms such
6,such as,such roberta
7,as roberta,roberta represent
8,roberta represent,represent text
9,represent text,text use


## 5. Observaciones

- Los auxiliares (`AUX`, p. ej. *is/are/were* → `be`) **no** están en la lista y sobreviven: hay que decidir si se quitan.
- Revisar en la tabla del paso 1 cómo quedan términos técnicos (siglas, palabras con guion).